# **2) Train Model**
Purpose : Multi-label genre classifier. The model uses TF-IDF and Logistic Regression to predict all genres that match a book based on its title and synopsis. Different model settings are tested using 5-fold cross-validation on the training data, and the test data is only used at the end to measure final performance.

Inputs  :
*   DATA/cleaned_book_data.csv   (from 1. Data Set Cleaning & EDA Script.py)



Outputs :
*   OUTPUT/model/tfidf_vectorizer.joblib
*   OUTPUT/model/logreg_model.joblib
*   OUTPUT/model/genre_classes.json
*   OUTPUT/test_predictions.csv
*   OUTPUT/cv_results.csv
*   OUTPUT/metrics_summary.json


Packages: pandas, numpy, scikit-learn, joblib

Run     : python SCRIPTS/2. Train Model.py   (from the repo root)

Order   : Run AFTER 1. Data Set Cleaning & EDA Script.py, BEFORE 3 Evaluate & Plot.py

In [1]:
# Anthropic. (2026). Claude Sonnet 5 [Large language model]. https://claude.ai/ Artificial intelligence was used to assist with code development and debugging.

import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, hamming_loss
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer

DATA_PATH = Path("cleaned_book_data.csv")
OUTPUT_DIR = Path("OUTPUT")
MODEL_DIR = OUTPUT_DIR / "model"

TEXT_COL = "text"        # built from Book Title + Synopsis below
GENRE_COL = "Genre"      # genre column
GENRE_SEP = ", "         # separates genres with ", " in the Genre column

RANDOM_SEED = 42         # fixed so the split, CV folds and model are reproducible
TEST_SIZE = 0.20         # 80/20 train/test split
N_FOLDS = 5              # 5-fold stratified cross-validation
MIN_GENRE_COUNT = 20     # safety net: drop genres in fewer books than this
THRESHOLD = 0.5          # probability cutoff for "book has this genre"

# Settings compared in cross-validation
PARAM_GRID = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],   # single words vs. words + word pairs
    "tfidf__max_features": [10000, 20000],    # limit on number of features
    "clf__estimator__C": [1.0, 5.0, 10.0],    # regularization (higher C = weaker)
}

OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# Helpers
def hit_rate(y_true, y_score):
    """Share of books whose single highest-scoring genre is one of their true genres."""
    top1 = np.argmax(y_score, axis=1)
    return float(y_true[np.arange(len(y_true)), top1].mean())


def top_k_hit_rate(y_true, y_score, k=3):
    """Share of books where at least one of the k highest-scoring genres is a true genre."""
    topk = np.argsort(y_score, axis=1)[:, -k:]
    hits = np.take_along_axis(y_true, topk, axis=1).max(axis=1)
    return float(hits.mean())


def threshold_predictions(y_score, threshold):
    """Predict every genre with probability >= threshold.
    If a book gets no genre, fall back to its single highest-scoring genre
    so every book receives at least one prediction."""
    y_pred = (y_score >= threshold).astype(int)
    empty = y_pred.sum(axis=1) == 0
    y_pred[empty, np.argmax(y_score[empty], axis=1)] = 1
    return y_pred


def cv_hit_rate_scorer(estimator, X, y):
    """Scorer used to choose settings in cross-validation (hit rate)."""
    return hit_rate(np.asarray(y), estimator.predict_proba(X))


# Load data
df = pd.read_csv(DATA_PATH).dropna(subset=["Book Title", "Synopsis", GENRE_COL])
df[TEXT_COL] = df["Book Title"].fillna("") + ". " + df["Synopsis"]  # combine title + synopsis
df["genre_list"] = df[GENRE_COL].apply(
    lambda s: [g.strip() for g in s.split(GENRE_SEP) if g.strip()]
)
print(f"Loaded {len(df)} books")

# Safety net: drop genres that are too rare to split or evaluate reliably
genre_counts = pd.Series([g for gl in df["genre_list"] for g in gl]).value_counts()
keep_genres = set(genre_counts[genre_counts >= MIN_GENRE_COUNT].index)
df["genre_list"] = df["genre_list"].apply(lambda gl: [g for g in gl if g in keep_genres])
df = df[df["genre_list"].str.len() > 0].reset_index(drop=True)
genre_counts = pd.Series([g for gl in df["genre_list"] for g in gl]).value_counts()
print(f"After safety filter: {len(df)} books, {len(genre_counts)} genres, "
      f"{df['genre_list'].str.len().mean():.2f} genres per book on average")

# Turn each book's genre list into a 0/1 row: one column per genre
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["genre_list"])
classes = mlb.classes_
print(f"Label matrix shape: {Y.shape}")

# Train/Test Split
# Standard stratification needs one label per book, but books have several.
# Workaround: stratify on each book's RAREST genre, which keeps rare genres
# represented in both train and test sets.
rarest_genre = df["genre_list"].apply(lambda gl: min(gl, key=lambda g: genre_counts[g]))

idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=rarest_genre,
)
X_train, X_test = df[TEXT_COL].values[idx_train], df[TEXT_COL].values[idx_test]
Y_train, Y_test = Y[idx_train], Y[idx_test]
print(f"Train size: {len(idx_train)} | Test size: {len(idx_test)}")

# Cross-validation
# The TF-IDF step lives INSIDE the pipeline, so in each fold it is fit only on
# that fold's training portion (no leakage into the validation portion).
# class_weight="balanced" upweights rare genres so they aren't ignored.
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english",
                              min_df=2, max_df=0.9, sublinear_tf=True)),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced",
                           random_state=RANDOM_SEED))),
])

# Stratified folds, again using each book's rarest genre as the stratifying label
cv_splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
cv_folds = list(cv_splitter.split(X_train, rarest_genre.values[idx_train]))

search = GridSearchCV(
    pipeline,
    param_grid=PARAM_GRID,
    scoring=cv_hit_rate_scorer,   # choose settings by hit rate (our accuracy measure)
    cv=cv_folds,
    n_jobs=-1,
    refit=True,                   # retrain the best setting on the FULL training set
    verbose=1,
)
search.fit(X_train, Y_train)

print(f"Best CV hit rate: {search.best_score_:.4f}")
print(f"Best settings   : {search.best_params_}")
pd.DataFrame(search.cv_results_).sort_values("rank_test_score").to_csv(
    OUTPUT_DIR / "cv_results.csv", index=False)

best_model = search.best_estimator_
vectorizer = best_model.named_steps["tfidf"]
clf = best_model.named_steps["clf"]

# Final Test Evaluation
# The test set is used exactly once.
Y_prob = best_model.predict_proba(X_test)
Y_pred = threshold_predictions(Y_prob, THRESHOLD)

results = {
    # Primary measure for the 85% goal: top predicted genre is a true genre
    "hit_rate_top1": hit_rate(Y_test, Y_prob),
    "hit_rate_top3": top_k_hit_rate(Y_test, Y_prob, k=3),
    # Strict measure: predicted genre SET must equal the true genre set
    "exact_match_accuracy": float((Y_pred == Y_test).all(axis=1).mean()),
    "f1_micro": f1_score(Y_test, Y_pred, average="micro", zero_division=0),
    "f1_macro": f1_score(Y_test, Y_pred, average="macro", zero_division=0),
    "f1_samples": f1_score(Y_test, Y_pred, average="samples", zero_division=0),
    "hamming_loss": hamming_loss(Y_test, Y_pred),
}

# Baseline: always predict the most common genre
most_common_idx = int(np.argmax(Y_train.sum(axis=0)))
baseline_hit = float(Y_test[:, most_common_idx].mean())

print("\n----- TEST SET RESULTS -----")
for name, val in results.items():
    print(f"{name:22s}: {val:.4f}")
print(f"{'baseline_hit_rate':22s}: {baseline_hit:.4f} (always '{classes[most_common_idx]}')")

# Save Results
joblib.dump(vectorizer, MODEL_DIR / "tfidf_vectorizer.joblib")
joblib.dump(clf, MODEL_DIR / "logreg_model.joblib")
with open(MODEL_DIR / "genre_classes.json", "w") as f:
    json.dump(list(classes), f, indent=2)

np.savez(OUTPUT_DIR / "test_scores.npz",
         y_true=Y_test, y_prob=Y_prob, y_pred=Y_pred, classes=classes)

pd.DataFrame({
    "text": X_test,
    "true_genres": [GENRE_SEP.join(classes[row == 1]) for row in Y_test],
    "predicted_genres": [GENRE_SEP.join(classes[row == 1]) for row in Y_pred],
    "top_predicted_genre": classes[np.argmax(Y_prob, axis=1)],
}).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

summary = {
    "n_books": int(len(df)),
    "n_train": int(len(idx_train)),
    "n_test": int(len(idx_test)),
    "n_genres": int(len(classes)),
    "avg_genres_per_book": round(float(Y.sum(axis=1).mean()), 3),
    "random_seed": RANDOM_SEED,
    "cv_folds": N_FOLDS,
    "prediction_threshold": THRESHOLD,
    "best_cv_hit_rate": round(float(search.best_score_), 4),
    "best_params": {k: str(v) for k, v in search.best_params_.items()},
    "test_results": {k: round(float(v), 4) for k, v in results.items()},
    "baseline_hit_rate": round(baseline_hit, 4),
    "baseline_genre": str(classes[most_common_idx]),
}
with open(OUTPUT_DIR / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved models, predictions, scores, and metrics to OUTPUT/")

Loaded 6609 books
After safety filter: 6609 books, 28 genres, 1.70 genres per book on average
Label matrix shape: (6609, 28)
Train size: 5287 | Test size: 1322
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best CV hit rate: 0.7416
Best settings   : {'clf__estimator__C': 5.0, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 1)}

----- TEST SET RESULTS -----
hit_rate_top1         : 0.7443
hit_rate_top3         : 0.8812
exact_match_accuracy  : 0.3971
f1_micro              : 0.6478
f1_macro              : 0.4776
f1_samples            : 0.6428
hamming_loss          : 0.0414
baseline_hit_rate     : 0.3321 (always 'Science Fiction')

Saved models, predictions, scores, and metrics to OUTPUT/
